# 📝 Day 3: Evaluation & Optimization
## Agentic RAG: From Zero to Hero

**:** 1 

## 🎓 Student Information


In [ ]:
# ─── ───
STUDENT_NAME = '' # ' '
STUDENT_ID = '' # '6512345678'
PHONE = '' # '081-234-5678'
LINE_ID = '' # 'somchai.j'

# ─── () ───
assert len(STUDENT_ID) >= 5, '❌ !'
assert len(STUDENT_NAME) >= 2, '❌ -!'

print(f'✅ -: {STUDENT_NAME}')
print(f'✅ : {STUDENT_ID}')
print(f'📱 : {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

---
## 📦 Dependencies

In [ ]:
%%time
!pip install -q ragas datasets google-adk google-genai \
 sentence-transformers qdrant-client langchain-text-splitters rank_bm25

import os, json, hashlib, random, asyncio, warnings
import numpy as np
warnings.filterwarnings('ignore')

try:
 from google.colab import userdata
 os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except:
 os.environ['GOOGLE_API_KEY'] = input('🔑 API Key: ')
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

from google import genai
client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
print('✅ Setup complete')

## 📄 ()

In [ ]:
%%time
# ─── Anti-Cheat: ───
seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (10**9)
rng = random.Random(seed)

all_topics = {
 'healthcare': [
 ' AI X-ray 95% 30 5 ',
 'NLP 15 2 ',
 ' AI Deep Learning ',
 ],
 'banking': [
 ' RAG Call Center 40% 24 ',
 ' Machine Learning real-time 60%',
 'AI ',
 ],
 'education': [
 ' RAG - 500 24 ',
 'Intelligent Tutoring System adaptive learning algorithm',
 'AI 70% 88% ',
 ],
 'agriculture': [
 'Smart Farming AI 8 92%',
 ' IoT sensor 85%',
 'AI Social Media ',
 ],
 'logistics': [
 ' AI 25% Graph Neural Network',
 ' AGV Computer Vision ',
 'Chatbot NLP real-time',
 ],
}

topics = rng.sample(list(all_topics.keys()), 4)
my_data = {}
for t in topics:
 my_data[t] = rng.sample(all_topics[t], 2)

print(f'📋 Topics ({STUDENT_ID}):')
for t, texts in my_data.items():
 print(f' 📌 {t}: {len(texts)} texts')
 for tx in texts:
 print(f' → {tx[:50]}...')

---
## 🎯 1: RAG Pipeline (3 )

 pipeline `my_data`:
1. Chunk RecursiveCharacterTextSplitter
2. Embed multilingual-e5-large
3. Upsert Qdrant
4. `search_qdrant(query)` + `rag_answer(question)`

In [ ]:
%%time
# ─── 1: RAG Pipeline ───
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 💡 Hint:
# 1. embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
# 2. splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
# 3. my_data → chunk → embed → list
# 4. qdrant = QdrantClient(':memory:') → create_collection → upsert
# 5. search_qdrant(query, top_k=3) → return [{text, source, score}]
# 6. rag_answer(question) → search + Gemini → return answer

# embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
# splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
#
# all_chunks, all_sources = [], []
# for src, texts in my_data.items():
# for text in texts:
# for chunk in splitter.split_text(text):
# all_chunks.append(chunk)
# all_sources.append(src)
#
# def search_qdrant(query, top_k=3):
# ...
#
# def rag_answer(question, top_k=3):
# ...


---
## 🎯 2: RAG (2.5 )

1. eval questions 5 ()
2. **LLM-as-Judge** 1-5
3. 

In [ ]:
%%time
# ─── 2: ───

# 💡 Hint:
# 1. eval_questions = ['1', '2', ...] 5 
# 2. rag_answer() answers
# 3. llm_judge() 
# 4. 

def llm_judge(question, answer):
 prompt = f''' 1-5:
Q: {question}
A: {answer}
 JSON: {{"score": 1-5, "reason": "..."}}'''
 resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
 config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json'))
 return json.loads(resp.text)

# eval_questions = [
# '...',
# '...',
# ]
# scores = []
# for q in eval_questions:
# ans, _ = rag_answer(q)
# verdict = llm_judge(q, ans)
# scores.append(verdict['score'])
# print(f" {'⭐'*verdict['score']} {q[:30]}...")
# print(f'Average: {sum(scores)/len(scores):.1f}/5.0')


---
## 🎯 3: Pipeline (2.5 )

1. **2 ** ( chunk_size + prompt)
2. **Before/After** LLM-as-Judge
3. **** 

In [ ]:
%%time
# ─── 3: ───

# 💡 Hint:
# 1. baseline_avg 2
# 2. chunk_size 100→300 → re-embed → re-search → 
# 3. prompt rag_answer " "
# 4. before vs after

# baseline_avg = ... # 2
# 
# # 1: chunk_size
# # 2: prompt
# 
# improved_avg = ...
# print(f'Before: {baseline_avg:.1f} → After: {improved_avg:.1f}')
# print(f': ...')


---
## 🎯 4: Agent + Test (2 )

1. Agent Google ADK Tool RAG
2. test cases **5 **
3. pass rate

In [ ]:
%%time
# ─── 4: Agent + Testing ───
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# 💡 Hint:
# 1. tool search_qdrant()
# def search_knowledge(query: str) -> dict:
# results = search_qdrant(query, top_k=3)
# return {'results': results}
#
# 2. Agent
# my_agent = Agent(name='hw3', model='gemini-2.5-flash',
# instruction='...', tools=[search_knowledge])
#
# 3. test cases
# test_cases = [
# {'input': '...', 'expected_tool': 'search_knowledge'},
# {'input': '...', 'expected_tool': None},
# ]
#
# 4. test pass rate


---
## 📊 

| | | |
|---------|:-----:|------|
| 1. RAG Pipeline | 3 | Pipeline , , rag_answer |
| 2. | 2.5 | eval questions ≥5, LLM-as-Judge |
| 3. | 2.5 | ≥2 , Before/After, |
| 4. Agent + Test | 2 | Agent , test cases ≥5, pass rate |
| **** | **10** | |

## ✅ Verify Submission


In [ ]:
# ─── () ───
print('═' * 50)
print(f'📋 Day 3')
print(f'═' * 50)
print(f': {STUDENT_NAME}')
print(f': {STUDENT_ID}')
print(f'Topics: {list(my_data.keys())}')

checks = {
 'RAG Pipeline': 'search_qdrant' in dir() or 'search_qdrant' in globals(),
 'rag_answer': 'rag_answer' in dir() or 'rag_answer' in globals(),
 'llm_judge': 'llm_judge' in dir() or 'llm_judge' in globals(),
}

for name, ok in checks.items():
 print(f" {'✅' if ok else '❌'} {name}")

if all(checks.values()):
 print('\n🎉 !')
else:
 print('\n⚠️ ')